# Notebook 1 – Dataset Exploration & Error Category Analysis

This notebook performs a thorough exploratory data analysis (EDA) of the CodeSense error dataset.  
It covers: loading, cleaning, category distributions, concept mappings, difficulty breakdown, and text statistics.

**Dataset:** `../dataset/error_dataset.csv`  
**Records:** 2,000 Python error examples across 8 categories

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# ── Load the dataset ──────────────────────────────────────────────────────────
df = pd.read_csv('../dataset/error_dataset.csv')

print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nFirst 3 rows:')
df.head(3)

In [ ]:
# ── Basic statistics ─────────────────────────────────────────────────────────
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Data types:')
print(df.dtypes)

In [ ]:
# ── Error Category Distribution ───────────────────────────────────────────────
category_counts = df['error_category'].value_counts()
print('Error Category Counts:')
print(category_counts.to_string())
print(f'\nTotal categories: {len(category_counts)}')
print(f'Class balance: min={category_counts.min()}, max={category_counts.max()}')
print(f'Dataset is perfectly balanced: {category_counts.min() == category_counts.max()}')

In [ ]:
# ── Bar chart: Error category distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#ef4444','#f59e0b','#3b82f6','#8b5cf6','#10b981','#06b6d4','#f97316','#64748b']

# Bar chart
axes[0].bar(category_counts.index, category_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Error Category Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
axes[0].set_xlabel('Error Category')
axes[0].tick_params(axis='x', rotation=35)
for i, v in enumerate(category_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', va='bottom', fontsize=10, fontweight='bold')

# Pie chart
axes[1].pie(
    category_counts.values,
    labels=category_counts.index,
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.75
)
axes[1].set_title('Category Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('category_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: category_distribution.png')

In [ ]:
# ── Difficulty breakdown ──────────────────────────────────────────────────────
diff_counts = df['difficulty'].value_counts()
print('Difficulty Breakdown:')
print(diff_counts.to_string())

# Cross-tabulation: difficulty per category
cross = pd.crosstab(df['error_category'], df['difficulty'])
print('\nDifficulty × Category Cross-tab:')
print(cross.to_string())

In [ ]:
# ── Code snippet length statistics ───────────────────────────────────────────
df['code_length'] = df['code'].str.len()
df['code_lines'] = df['code'].str.count('\n') + 1

print('Code Length (characters):')
print(df.groupby('error_category')['code_length'].describe().round(1).to_string())
print()
print('Code Lines Count:')
print(df.groupby('error_category')['code_lines'].agg(['mean','min','max']).round(1).to_string())

In [ ]:
# ── Most common concepts per category ────────────────────────────────────────
print('Top 3 concepts per error category:')
for cat in sorted(df['error_category'].unique()):
    concepts = df[df['error_category'] == cat]['concept'].value_counts().head(3)
    print(f'\n{cat}:')
    for concept, count in concepts.items():
        print(f'  • {concept} ({count})')

In [ ]:
# ── Keyword frequency analysis ────────────────────────────────────────────────
all_keywords = ' '.join(df['keywords'].dropna().tolist()).lower().split()
keyword_counts = Counter(all_keywords)
print('Top 20 Most Frequent Keywords:')
for kw, count in keyword_counts.most_common(20):
    bar = '█' * (count // 5)
    print(f'  {kw:20s} {count:4d}  {bar}')

In [ ]:
# ── Summary insights ──────────────────────────────────────────────────────────
print('=== Dataset Summary ===')
print(f'Total records     : {len(df):,}')
print(f'Error categories  : {df["error_category"].nunique()}')
print(f'Unique concepts   : {df["concept"].nunique()}')
print(f'Languages         : {df["language"].unique()}')
print(f'Difficulty levels : {df["difficulty"].unique()}')
print(f'Avg code length   : {df["code_length"].mean():.0f} chars')
print(f'Missing values    : {df.isnull().sum().sum()}')
print(f'Perfectly balanced: {df["error_category"].value_counts().std() == 0}')